# STEM Audio Workbench — Website-connected Colab Bridge\n\nThis notebook turns your temporary Google Colab runtime into the STEM processing engine for the public website. You upload and view results on the website; Colab does the actual AI processing.\n\n## Start it\n1. Open this notebook in Google Colab and choose **Runtime → Run all**.\n2. Wait for the cell to print the temporary HTTPS bridge address and session token.\n3. Open the [STEM website](https://demtaiwo1-eng.github.io/stem.github.io/), paste both values into **Connect your Colab processing session**, and press **Connect**.\n4. Upload your song on the website. Processing progress, audio previews, individual WAV/MP3 downloads and the full ZIP will appear there.\n\nKeep this notebook runtime running until your results are downloaded. Colab may disconnect, sleep or expire; when it does, run this notebook again and reconnect with the new address and token. Free GPU availability is controlled by Google; CPU fallback is slower. Only upload audio you own or are authorised to process.\n

In [ ]:
!apt-get -qq update\n!apt-get -qq install -y ffmpeg\n!pip -q install demucs==4.0.1 flask flask-cors\n!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared\n!chmod +x /usr/local/bin/cloudflared\n!ffmpeg -version | head -n 1\n!python -m demucs --help > /dev/null && echo 'Demucs installation OK'\n!cloudflared --version\n

In [ ]:
import os, sys, shutil, subprocess, json, uuid, secrets, re, threading, queue, time\nfrom pathlib import Path\nimport torch\n\nREPO = Path('/content/STEM')\nif REPO.exists(): shutil.rmtree(REPO)\nsubprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/demtaiwo1-eng/Stem.git', str(REPO)], check=True)\nif not (REPO / 'worker.py').is_file() or not (REPO / 'colab_bridge.py').is_file():\n    raise RuntimeError('The STEM worker or Colab bridge was not found in the repository.')\nDEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'\nTOKEN = secrets.token_urlsafe(28)\nenv = os.environ.copy()\nenv['STEM_DEVICE'] = DEVICE\nenv['STEM_MODEL'] = 'htdemucs_6s'\nenv['STEM_VOCAL_FALLBACK_MODEL'] = 'htdemucs'\nenv['STEM_BRIDGE_TOKEN'] = TOKEN\nbridge_log = open('/content/stem-bridge.log', 'w')\nbridge = subprocess.Popen([sys.executable, str(REPO / 'colab_bridge.py')], cwd=str(REPO), env=env, stdout=bridge_log, stderr=subprocess.STDOUT)\ntime.sleep(3)\nif bridge.poll() is not None:\n    raise RuntimeError('STEM bridge failed to start. Check /content/stem-bridge.log')\nprint('STEM engine is starting on', DEVICE)\nif DEVICE == 'cpu': print('No GPU is attached. Processing will use CPU and may be slow.')\nevents = queue.Queue()\ndef read_tunnel(pipe):\n    for line in iter(pipe.readline, ''): events.put(line.rstrip())\ntunnel = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8787', '--no-autoupdate'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)\nthreading.Thread(target=read_tunnel, args=(tunnel.stdout,), daemon=True).start()\nPUBLIC_URL = None\ndeadline = time.time() + 120\nwhile time.time() < deadline and PUBLIC_URL is None:\n    try: line = events.get(timeout=2)\n    except queue.Empty: continue\n    print(line)\n    match = re.search(r'https://[a-z0-9-]+\\.trycloudflare\\.com', line)\n    if match: PUBLIC_URL = match.group(0)\nif not PUBLIC_URL:\n    tunnel.terminate()\n    raise RuntimeError('Could not obtain a Cloudflare temporary URL. Re-run this cell or restart the Colab runtime.')\nprint('\\n' + '='*58)\nprint('STEM COLAB BRIDGE IS READY')\nprint('Temporary website connection:', PUBLIC_URL)\nprint('Session token:', TOKEN)\nprint('Paste both values into the STEM website and press Connect.')\nprint('Keep this Colab runtime running while you upload and process songs.')\nprint('='*58)\n

In [ ]:
print('Bridge status:')\nprint(subprocess.check_output(['curl', '-s', 'http://127.0.0.1:8787/health'], text=True))\nprint('The website can now upload audio to this Colab session. Leave this notebook open while jobs run.')\nprint('If the website reports a disconnected session, restart the bridge cell and reconnect with the newly printed URL and token.')\n

In [ ]:
print('The STEM website now handles uploads, progress, previews and downloads.')\nprint('Open: https://demtaiwo1-eng.github.io/stem.github.io/')\nprint('Do not stop this runtime until all results have been downloaded.')\n